# Extract Scalar Performance of models from a log file

## Fichiers

+ E13sg_8-9_run_all_agents_test.log, expe version minor 8 et 9 AVEC (!) la latence dans les observations, Randtype Gaussian
+ E13sg_10_run_all_agents_test.log, expe SANS la latence dans les observations, Randtype Gaussian 
+ E13sgu_11-12_a2c_run_all_agents_test.log, expe SANS la latence dans les observations, Randtypes Gaussian+Uniform 
+ E13sgu_11-12_run_all_agents_test.log
+ E15sall_14_run_all_agents_test.log, expe incluant datasets v9, v10, v12.1 up to v12.19. Randtypes: all, Bug on CVAE dataset wich is reducesed because of wl_client at 2.999
+ E15sall_15_run_all_agents_test.log, expe incluant datasets v9, v10, v12.1 up to v12.19. Randtypes: all, complete datasets, fixed 1000ms maxlimit for ilat computation 

In [1]:
# Chemin vers votre fichier de log
log_file_path = 'results/E15sall_15_run_all_agents_test.log'

In [2]:
import os
import re
import pandas as pd

# Listes pour stocker les données extraites
data = []

# Expressions régulières
pattern_result = re.compile(
    r'Trial:\s*(\d+).*?TestPerf\(ScalarPerf,USLA,RAM,SLAV\):\s*\(([^,]+),\s*([^,]+),\s*([^,]+),\s*([^\)]+)\)'
)

pattern_agent = re.compile(
    r'AgentFilename:\s*\.\/outputs\/([^\/]+)\/agent-([^_]+)-([^_]+)_'
)

# Variable pour stocker temporairement les infos Agent
current_agent_info = None

# Fonction pour extraire "version" et "simu" à partir du nom de dossier
def extract_version_simu(folder_name):
    # Exemple : E13sg_9_tabddpm-a2c-training_disq
    parts = folder_name.split('_')
    if len(parts) >= 3:
        version = parts[0]
        version_minor = parts[1]  # E13sg_9
        simu = parts[2].split('-')[0]     # tabddpm
        return version, version_minor, simu
    return None, None, None

line1 = None
line2 = None

# Lecture du fichier de log
with open(log_file_path, 'r') as file:
    for line in file:
        if line1 is None and line.startswith('Version:'):
            line1 = line.strip()
        # Vérifier la deuxième ligne
        elif line2 is None and line.startswith('Description -'):
            line2 = line.strip()

        # Vérifier si la ligne est une ligne AgentFilename
        match_agent = pattern_agent.search(line)
        if match_agent:
            folder_name = match_agent.group(1)
            version, version_minor, simu = extract_version_simu(folder_name)
            model = match_agent.group(3)
            # Stocker temporairement ces infos
            current_agent_info = {
                'Version': version,
                'VersionMinor': version_minor,
                'Simu': simu,
                'Model': model
            }
        # Vérifier si la ligne est une ligne TestPerf
        match_result = pattern_result.search(line)
        if match_result:
            trial_num = int(match_result.group(1))
            scalar_perf = float(match_result.group(2))
            usla = int(match_result.group(3))
            ram = int(float(match_result.group(4)))
            slav = int(match_result.group(5))
            # Créer l'entrée avec les résultats
            entry = {
                'Trial': trial_num,
                'ScalarPerf': scalar_perf,
                'USLA': usla,
                'RAM': ram,
                'SLAV': slav,
                'Version': None,
                'VersionMinor': None,
                'Simu': None,
                'Model': None
            }
            # Ajouter les infos Agent si disponibles
            if current_agent_info:
                entry.update(current_agent_info)
            data.append(entry)

# Convertir en DataFrame
dffull = pd.DataFrame(data)

base_name, _ = os.path.splitext(log_file_path)
expcsv=base_name+".csv"
print(f"Save results dataset to: {expcsv}...")
dffull.to_csv(expcsv, index=False)

# Afficher le DataFrame final
#print(df)
version_list = dffull['Version'].unique()
version_minor_list = dffull['VersionMinor'].unique()
simu_list = dffull['Simu'].unique()
model_list = dffull['Model'].unique()
print("Versions found in the log file:")
print(version_list)
print(version_minor_list)
print(simu_list)
print(model_list)
print("")
print(line1)
print(line2)
print("")
roundat=3

pd.options.display.width = 500

# Generate result table
for version in version_list:
    results = []
    for version_minor in version_minor_list:
        for simu in simu_list:
            for model in model_list:
                dfperf = dffull.loc[(dffull['Version'] == version) & (dffull['VersionMinor'] == version_minor) & (dffull['Simu'] == simu) & (dffull['Model'] == model)]
                if not dfperf.empty:
                    #dfperf = dffull.loc[(dffull['Version'] == version) & (dffull['VersionMinor'] == version_minor) & (dffull['Simu'] == simu) & (dffull['Model'] == model)]
                    scalar_perf_min = dfperf['ScalarPerf'].min()
                    scalar_perf_max = dfperf['ScalarPerf'].max()
                    dfmin = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_min]
                    dfmax = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_max]

                    scalar_perf_max = round(scalar_perf_max, roundat)
                    scalar_perf_min = round(scalar_perf_min, roundat)
                    scalar_perf_mean = round(dfperf['ScalarPerf'].mean(), roundat)
                    scalar_perf_std = round(dfperf['ScalarPerf'].std(), roundat)
                    entry = {
                        'Version': version+"_"+version_minor,
                        'Dataset': str(simu).upper(),
                        'Model': model,
                        'Perfmean': scalar_perf_mean,
                        'Perfminmax': str(scalar_perf_min)+"/"+str(scalar_perf_max),
                        'Perfstd': scalar_perf_std,
                        'USLA@Perfminmax': str(dfmin.iloc[0]["USLA"])+"/"+str(dfmax.iloc[0]["USLA"]),
                        'RAM@Perfminmax(MB)': str(dfmin.iloc[0]["RAM"]//(1024*1024))+"/"+str(dfmax.iloc[0]["RAM"]//(1024*1024)),
                        'SLAV@Perfminmax': str(dfmin.iloc[0]["SLAV"])+"/"+str(dfmax.iloc[0]["SLAV"]),
                    }
                    results.append(entry)

    # Convertir en DataFrame
    dfresults = pd.DataFrame(results)
    dfresults = dfresults.sort_values(by=['Version', 'Dataset', 'Perfmean'], ascending=[True, False, True])

    # Afficher le DataFrame final
    print(dfresults.to_string(index=False))


Save results dataset to: results/E15sall_15_run_all_agents_test.csv...
Versions found in the log file:
['E15sall']
['15']
['cvae' 'tabddpm' 'dbsas' 'orig']
['SB3DQN' 'SB3A2C' 'SB3PPO']

Version: 15sall, minor versions: 15, datasets: orig dbsas cvae tabddpm, models: ppo a2c dqn, experiments: training
Description - 15sall_15: orig+dbsas+cvae+tabddpm

   Version Dataset  Model  Perfmean    Perfminmax  Perfstd USLA@Perfminmax RAM@Perfminmax(MB) SLAV@Perfminmax
E15sall_15 TABDDPM SB3DQN     3.820   1.208/8.372    2.533      4683/22264             147/93      1366/21289
E15sall_15 TABDDPM SB3PPO    10.136  9.461/11.241    0.542     24669/28862              82/76     24669/28861
E15sall_15 TABDDPM SB3A2C    13.419 13.418/13.419    0.000     33675/33677              54/54     33675/33677
E15sall_15    ORIG SB3DQN     0.553   0.344/0.785    0.144       2665/3911            144/135        165/1418
E15sall_15    ORIG SB3A2C     0.701   0.459/0.935    0.167       2593/3937            160/151      